# GQA + QK-Norm + RoPE 因果自注意力

源码导航：[`core/attention/walkie_attention.py`](../../../core/attention/walkie_attention.py) 中的 `repeat_kv`、`WalkieCausalSelfAttention`。

标准多头自注意力（MHA）的每层计算复杂度与 KV cache 占用均正比于头数。随着序列长度和批大小扩大，KV cache 往往成为推理的显存瓶颈。Walkie 的注意力模块集成了三项现代改进：

- **GQA (Grouped Query Attention)**：$n_{\text{head}}$ 个 Q 头共享 $n_{\text{head\_kv}}$ 个 KV 头（$n_{\text{head\_kv}} \ll n_{\text{head}}$），KV cache 缩减至 $n_{\text{head\_kv}} / n_{\text{head}}$ 倍。
- **QK-Norm**：对每头的 Q/K 各做一次 RMSNorm，防止注意力 logits 在训练初期因维度缩放爆炸导致 softmax 饱和。
- **RoPE**：在 Q/K 上注入旋转位置编码，点积后自然带有相对位置信息，无需修改 token embedding。

### 1. 理论推导

标准因果注意力：

$$
\operatorname{Attn}(Q, K, V) = \operatorname{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_h}} + M_{\text{causal}}\right) V
$$

其中 $d_h = $ `head_dim`，$M_{\text{causal}}$ 是上三角负无穷掩码。

**GQA 的 KV 展开**：每个 KV 头被 $n_{\text{rep}} = n_{\text{head}} / n_{\text{head\_kv}}$ 个 Q 头共享，通过 `repeat_kv` 将形状从 $(B, H_{kv}, T, d_h)$ 扩展为 $(B, H_q, T, d_h)$：

```python
def repeat_kv(x, n_rep):
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)
```

**完整 forward 数据流**：
1. $Q = \operatorname{Proj}_Q(x)$，形状 $(B, H_q, T, d_h)$
2. $K, V = \operatorname{Proj}_{K/V}(x)$，形状 $(B, H_{kv}, T, d_h)$
3. $Q \leftarrow \operatorname{RMSNorm}_Q(Q)$，$K \leftarrow \operatorname{RMSNorm}_K(K)$（QK-Norm，每头独立）
4. $Q, K \leftarrow \operatorname{RoPE}(Q, K)$（旋转位置编码）
5. $K, V \leftarrow \operatorname{repeat\_kv}(K, n_{\text{rep}})$（GQA 展开）
6. 计算因果注意力，输出 $y$，形状 $(B, T, H_q \cdot d_h)$
7. $\operatorname{out} = \operatorname{Proj}_O(y)$，映射回 $n_{\text{embd}}$

### 2. 实现路径

Walkie 支持三种注意力后端，由 `attn_impl` 控制：

| `attn_impl` | 描述 |
|---|---|
| `sdpa` | PyTorch `scaled_dot_product_attention`，默认路径，支持 FlashAttention 内核 |
| `flash_attn2` | 显式调用 `flash_attn` 库，需要 CUDA + 半精度，延迟更低 |
| `eager` | 手动矩阵乘法，保留完整中间张量，用于教学与对齐调试 |

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.walkie_attention import WalkieCausalSelfAttention, repeat_kv

### 3. GQA 的 KV head 展开验证

In [ ]:
import torch
kv = torch.randn(2, 2, 8, 16)  # (B=2, H_kv=2, T=8, D=16)
expanded = repeat_kv(kv, n_rep=4)  # n_rep = n_head / n_head_kv = 8/2 = 4

print("kv.shape      :", tuple(kv.shape))
print("expanded.shape:", tuple(expanded.shape))  # (2, 8, 8, 16)

# 验证每个 KV 头的内容被复制 4 次
# head 0, 1, 2, 3 都应等于原 kv 的 head 0
for i in range(4):
    assert torch.allclose(expanded[:, i, :, :], kv[:, 0, :, :]), f"head {i} 展开不匹配！"
print("每个 KV head 均被正确扩展 4 次。")

### 4. 前向传播与投影维度验证

In [ ]:
attn = WalkieCausalSelfAttention(
    n_embd=128,
    n_head=8,        # Q 头数
    n_head_kv=2,     # KV 头数，n_rep = 8/2 = 4
    head_dim=16,     # 每头维度，q_dim = 8*16 = 128，kv_dim = 2*16 = 32
    max_seq_len=64,
    dropout=0.0,
    qk_norm=True,    # 启用 QK-Norm
    attn_impl='sdpa',
)

x = torch.randn(2, 12, 128)  # (B=2, T=12, n_embd=128)
y = attn(x)

print("output:", tuple(y.shape))  # 应为 (2, 12, 128)
print("\n各投影矩阵维度 (out_features, in_features):")
for name in ['q_proj', 'k_proj', 'v_proj', 'o_proj']:
    mod = getattr(attn, name)
    print(f"  {name:6s}: {tuple(mod.weight.shape)}")

print("\nQK-Norm 参数维度:")
print(f"  q_norm.weight: {tuple(attn.q_norm.weight.shape)}")  # (head_dim,) = (16,)
print(f"  k_norm.weight: {tuple(attn.k_norm.weight.shape)}")

### 5. 源码精讲：forward 数据流

```python
def forward(self, x: torch.Tensor) -> torch.Tensor:
    B, T, _ = x.shape

    # 1. 线性投影 Q/K/V（无 bias）
    q = self.q_proj(x).view(B, T, self.n_head,    self.head_dim).transpose(1, 2)  # (B, H_q, T, d_h)
    k = self.k_proj(x).view(B, T, self.n_head_kv, self.head_dim).transpose(1, 2)  # (B, H_kv, T, d_h)
    v = self.v_proj(x).view(B, T, self.n_head_kv, self.head_dim).transpose(1, 2)

    # 2. QK-Norm：对每头的 head_dim 维独立做 RMSNorm
    if self.q_norm is not None:
        q = self.q_norm(q)
        k = self.k_norm(k)

    # 3. RoPE：在头维度上施加旋转，注入相对位置信息
    cos, sin = self.rope(T, device=x.device, dtype=q.dtype)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    # 4. SDPA 路径：展开 KV 头后调用 PyTorch 内核
    k = repeat_kv(k, self.n_rep)   # (B, H_q, T, d_h)
    v = repeat_kv(v, self.n_rep)
    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    y = y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)

    # 5. 输出投影，映射回 n_embd
    return self.resid_dropout(self.o_proj(y))
```

注意，`q_norm` / `k_norm` 的 `normalized_shape` 为 `head_dim`（而非 `n_embd`），这意味着归一化是**按头独立**进行的，每头有各自的 $w$ 向量，参数量为 $2 \times n_{\text{head}} \times d_h = 2 \times n_{\text{embd}}$，相较于全量 Q/K 矩阵可忽略不计。

---

## 延伸阅读与参考资料

### 核心论文
- **Attention Is All You Need**: Vaswani et al., 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- **GQA: Training Generalized Multi-Query Transformer Models**: Ainslie et al., 2023. [arXiv:2305.13245](https://arxiv.org/abs/2305.13245)
- **FlashAttention**: Dao et al., 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)

### 工程实现
- **PyTorch SDPA**: [docs](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
- **Hugging Face LlamaAttention**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)